### Configuração e Importações

In [1]:
import pandas as pd
import numpy as np
import os

# Configurando o Pandas para mostrar todas as colunas
pd.set_option('display.max_columns', None)

# Definindo o caminho base (assumindo que o notebook está em notebooks/)
# Suba um nível (..) e entre na pasta data/raw
CAMINHO_RAW = '../data/raw/'

print("Bibliotecas importadas e caminhos definidos.")

Bibliotecas importadas e caminhos definidos.


### Carregando e Unindo os Dados (ETL)

In [3]:
# Célula 2: Carregando e Unindo os Dados (ETL)

try:
    # Lendo o arquivo FATO (Logs de Atendimento)
    df_logs = pd.read_csv(os.path.join(CAMINHO_RAW, 'logs_atendimento.csv'), sep=',', encoding='utf-8')
    
    # Lendo o arquivo DIMENSÃO (Escala de Médicos) - Forçando a robustez para leitura em Notebook
    df_medicos = pd.read_csv(os.path.join(CAMINHO_RAW, 'escala_medicos.csv'), 
                            sep=';', 
                            encoding='utf-8-sig',
                            engine='python', # Motor mais tolerante para Notebook
                            quoting=3)      # Desativa leitura de aspas

except FileNotFoundError as e:
    print(f"ERRO: Verifique se os arquivos estão em {CAMINHO_RAW}. Detalhe: {e}")
    
# --- ETAPA DE TRANSFORMAÇÃO: LIMPEZA ESSENCIAL DO CABEÇALHO ---

# Apenas limpa os nomes das COLUNAS do df_medicos (Resolve o KeyError)
df_medicos.columns = df_medicos.columns.str.strip().str.replace('"', '').str.replace(':', '')

# Unindo os dados (Join)
df_logs_completo = df_logs.merge(df_medicos[['id_medico', 'perfil_variabilidade']], 
                                on='id_medico', 
                                how='left')

# Exibindo as primeiras linhas do dataset unido
print(f"Total de Registros Carregados: {len(df_logs_completo):,}")
print("\n--- Primeiras Linhas (Data Bruta Unida) ---")
display(df_logs_completo.head())

Total de Registros Carregados: 2,000

--- Primeiras Linhas (Data Bruta Unida) ---


,id_atendimento,data_chegada,hora_checkin,hora_consulta_inicio,hora_consulta_fim,id_medico,status,perfil_variabilidade
0,1000,2025-10-30,11:20:48,17:33:31,17:53:31,M009,Desistiu,NaN
1,1001,2025-10-30,13:31:25,18:10:20,18:30:20,M017,Atendido,NaN
2,1002,2025-10-30,10:03:12,12:23:19,12:43:19,M008,Atendido,NaN
3,1003,2025-10-30,10:48:05,16:33:36,16:53:36,M006,Atendido,NaN
4,1004,2025-10-30,12:11:46,15:07:12,15:27:12,M016,Atendido,NaN


### Cálculo da Métrica Y (Transformação)

O tempo de espera é a nossa Métrica Y e o principal foco do DMAIC.

$$\text{Tempo de Espera} = \text{Hora Consulta Início} - \text{Hora Check-in}$$

In [4]:
# Convertendo as colunas de hora/data para o formato datetime para fazer cálculos
df_logs_completo['checkin_dt'] = pd.to_datetime(df_logs_completo['data_chegada'] + ' ' + df_logs_completo['hora_checkin'])
df_logs_completo['consulta_inicio_dt'] = pd.to_datetime(df_logs_completo['data_chegada'] + ' ' + df_logs_completo['hora_consulta_inicio'], errors='coerce')

# 1. Calcular a Métrica Y (Tempo de Espera em Minutos)
# O cálculo só é feito para pacientes que foram atendidos (não desistiram)
df_logs_completo['tempo_espera_total'] = (df_logs_completo['consulta_inicio_dt'] - df_logs_completo['checkin_dt']).dt.total_seconds() / 60

# 2. Calcular a Taxa de Evasão (KPI de Negócio)
taxa_evasao = df_logs_completo['status'].value_counts(normalize=True).get('Desistiu', 0) * 100

print("Transformação de tempo concluída.")
print(f"Taxa de Evasão (Desistência de Pacientes): {taxa_evasao:.2f}%")

Transformação de tempo concluída.
Taxa de Evasão (Desistência de Pacientes): 15.40%


### Análise Estatística (Prova do Problema)

In [6]:
# Filtrar apenas pacientes atendidos para calcular a Métrica Y (Tempo de Espera)
df_atendidos = df_logs_completo[df_logs_completo['status'] == 'Atendido'].copy()

# O alvo do DMAIC é provar a crise de 4 horas (240 minutos)
MEDIA_ALVO = 240 # Minutos (4 horas)

# Prova estatística da Fase MEASURE
media_espera_atual = df_atendidos['tempo_espera_total'].mean()
desvio_padrao = df_atendidos['tempo_espera_total'].std()
max_espera = df_atendidos['tempo_espera_total'].max()

print("\n--- Prova da Crise (Métrica Y) ---")
print(f"Média Alvo (4h em min): {MEDIA_ALVO} minutos")
print(f"MÉDIA ATUAL DE ESPERA: {media_espera_atual:.2f} minutos")
print(f"Desvio Padrão (Variabilidade): {desvio_padrao:.2f} minutos")
print(f"Máximo de Espera (Pico de Crise): {max_espera/60:.2f} horas ({max_espera:.2f} min)")

if media_espera_atual >= MEDIA_ALVO:
    print("\n CRÍTICO: A média de espera de 4 horas (240 min) está comprovada!")
    print("O projeto Green Belt 'GreenBelt' é estatisticamente necessário.")
else:
    print("\nSucesso: A média de espera está abaixo das 4 horas.")


--- Prova da Crise (Métrica Y) ---
Média Alvo (4h em min): 240 minutos
MÉDIA ATUAL DE ESPERA: 241.91 minutos
Desvio Padrão (Variabilidade): 123.52 minutos
Máximo de Espera (Pico de Crise): 8.33 horas (500.00 min)

 CRÍTICO: A média de espera de 4 horas (240 min) está comprovada!
O projeto Green Belt 'GreenBelt' é estatisticamente necessário.
